# Validate expression split

This notebook is for manual validation of the rule-based relational versus attribute split produced by `data/classify_expressions.py`. Following the project methodology, review an approximately balanced audit sample of about 100 expressions per predicted class, record whether each prediction is correct, and use the resulting judgments to estimate precision for the `relational` and `attribute` labels.

In [ ]:
from pathlib import Path

import pandas as pd

# Edit these before running the audit.
LOG_CSV_PATH = Path('data/splits/refcoco_val_classification_log.csv')
N_PER_LABEL = 100
RANDOM_SEED = 7

df = pd.read_csv(LOG_CSV_PATH)
sampled = (
    df.groupby('label', group_keys=False)
    .apply(lambda group: group.sample(n=min(N_PER_LABEL, len(group)), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

print(sampled['label'].value_counts().sort_index())
sampled.head()

In [ ]:
judgments = []

# Review one prediction at a time and record whether the predicted label is correct.
for row in sampled.itertuples(index=False):
    print()
    print(f"Expression: {row.expression}")
    print(f"Matched cue: {row.matched_cue}")
    print(f"Predicted label: {row.label}")

    while True:
        answer = input('Correct? [y/n]: ').strip().lower()
        if answer in {'y', 'yes', 'n', 'no'}:
            judgments.append(answer.startswith('y'))
            break
        print('Please answer with y or n.')

sampled = sampled.copy()
sampled['human_correct'] = judgments
sampled[['expression', 'matched_cue', 'label', 'human_correct']].head()

In [ ]:
from pathlib import Path

# Precision is the share of predicted examples judged correct within each label.
precision = sampled.groupby('label')['human_correct'].mean().sort_index()
for label, value in precision.items():
    print(f"{label}: {value:.3f}")

output_path = Path('results/expression_classifier_audit.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
sampled.to_csv(output_path, index=False)
print(f"Saved audit annotations to {output_path}")

Paste the resulting per-class precision numbers into the README `Limitations` section after finishing the audit.